## Example 1: ServerSentEvent creation and handling

In [ ]:
import httpx_sse

# Example 1: ServerSentEvent creation and handling
print("Example 1: ServerSentEvent creation")

# Create ServerSentEvent objects
event1 = httpx_sse.ServerSentEvent(data="Hello World", event="message")
print(f"Event 1 created: {type(event1).__name__}")
print(f"Event data: {event1.data}")
print(f"Event type: {event1.event}")
print(f"Event ID: {event1.id}")

# Create event with ID and retry
event2 = httpx_sse.ServerSentEvent(
    data="Test data", 
    event="test", 
    id="123",
    retry=5000
)
print(f"Event 2 - ID: {event2.id}, Retry: {event2.retry}")

# Create simple data-only event
event3 = httpx_sse.ServerSentEvent(data="Simple message")
print(f"Event 3 - Data: {event3.data}")

assert (event1.event, event1.data, event1.id, event1.retry) == ("message", "Hello World", "", None)
assert (event2.event, event2.id, event2.retry) == ("test", "123", 5000)
# An event without a type defaults to "message", as in the SSE specification
assert event3.event == "message"

## Example 2: Reading an event stream with connect_sse

In [ ]:
import httpx_sse
import httpx

# Example 2: Reading an event stream with connect_sse
print("Example 2: Reading an event stream with connect_sse")

# An in-process transport serves a fixed text/event-stream body, so no network is needed
STREAM = (
    b"event: greeting\ndata: hello\nid: 1\n\n"
    b'data: {"n": 2}\n\n'
    b"data: line1\ndata: line2\nretry: 3000\n\n"
)
seen_headers = {}

def handler(request):
    seen_headers.update(request.headers)
    return httpx.Response(200, headers={"content-type": "text/event-stream"}, content=STREAM)

with httpx.Client(transport=httpx.MockTransport(handler)) as client:
    with httpx_sse.connect_sse(client, "GET", "https://example.com/stream") as event_source:
        events = list(event_source.iter_sse())

for event in events:
    print(f"  {event!r}")

assert len(events) == 3
assert (events[0].event, events[0].data, events[0].id) == ("greeting", "hello", "1")
# The last event id carries over to later events that do not set one
assert events[1].event == "message" and events[1].id == "1"
assert events[1].json() == {"n": 2}
# Consecutive data lines are joined with a newline; retry is parsed as milliseconds
assert events[2].data == "line1\nline2" and events[2].retry == 3000
# connect_sse asks for an event stream and disables caching
assert seen_headers["accept"] == "text/event-stream"
assert seen_headers["cache-control"] == "no-store"

## Example 3: Reading an event stream asynchronously with aconnect_sse

In [ ]:
import httpx_sse
import httpx

# Example 3: Reading an event stream asynchronously with aconnect_sse
print("Example 3: Reading an event stream asynchronously with aconnect_sse")

async def async_handler(request):
    return httpx.Response(
        200,
        headers={"content-type": "text/event-stream"},
        content=b"event: tick\ndata: 1\n\nevent: tick\ndata: 2\n\nevent: done\ndata: \n\n",
    )

async def read_ticks():
    async with httpx.AsyncClient(transport=httpx.MockTransport(async_handler)) as async_client:
        async with httpx_sse.aconnect_sse(async_client, "GET", "https://example.com/ticks") as event_source:
            return [event async for event in event_source.aiter_sse()]

async_events = await read_ticks()
for event in async_events:
    print(f"  {event!r}")

assert [e.event for e in async_events] == ["tick", "tick", "done"]
assert [e.data for e in async_events] == ["1", "2", ""]

## Example 4: SSE error handling and event validation

In [ ]:
import httpx_sse

# Example 4: SSE error handling and event validation
print("Example 4: SSE error handling")

# Create SSE error for error handling demonstration
try:
    sse_error = httpx_sse.SSEError("Connection failed")
    print(f"SSE Error created: {type(sse_error).__name__}")
    print(f"Error message: {str(sse_error)}")
except Exception as e:
    print(f"Error creation failed: {e}")
    raise

# Test different error scenarios
error_scenarios = [
    "Connection timeout",
    "Invalid event format", 
    "Stream interrupted",
    "Authentication failed"
]

print("Testing error scenarios:")
for i, error_msg in enumerate(error_scenarios, 1):
    try:
        test_error = httpx_sse.SSEError(error_msg)
        print(f"  {i}. {test_error}")
    except Exception as e:
        print(f"  {i}. Error scenario: {error_msg} - {e}")
        raise

# Show SSEError inheritance
print(f"SSEError base classes: {httpx_sse.SSEError.__bases__}")
assert issubclass(httpx_sse.SSEError, httpx.TransportError)
assert str(sse_error) == "Connection failed"

# Reading events from a response that is not text/event-stream raises SSEError
import httpx

def json_handler(request):
    return httpx.Response(200, headers={"content-type": "application/json"}, content=b"{}")

with httpx.Client(transport=httpx.MockTransport(json_handler)) as client:
    with httpx_sse.connect_sse(client, "GET", "https://example.com/not-a-stream") as event_source:
        try:
            list(event_source.iter_sse())
            raise AssertionError("expected SSEError")
        except httpx_sse.SSEError as e:
            print(f"Rejected: {e}")
            assert "text/event-stream" in str(e)

## Example 5: SSE event parsing and data handling

In [ ]:
import httpx_sse

# Example 5: SSE event parsing and data handling
print("Example 5: SSE event parsing and data handling")

# Create events with different data formats
json_event = httpx_sse.ServerSentEvent(
    data='{"message": "Hello", "timestamp": 1234567890}',
    event="json-message",
    id="msg-001"
)

text_event = httpx_sse.ServerSentEvent(
    data="Simple text message",
    event="text-message"
)

multiline_event = httpx_sse.ServerSentEvent(
    data="Line 1\nLine 2\nLine 3",
    event="multiline"
)

events = [json_event, text_event, multiline_event]

# Process different event types
for i, event in enumerate(events, 1):
    print(f"Event {i}:")
    print(f"  Type: {event.event}")
    print(f"  Data: {event.data}")
    print(f"  ID: {event.id}")
    print(f"  Data length: {len(event.data) if event.data else 0}")

assert json_event.json() == {"message": "Hello", "timestamp": 1234567890}
assert json_event.id == "msg-001" and text_event.id == ""
assert multiline_event.data.splitlines() == ["Line 1", "Line 2", "Line 3"]